# MoneyPrinterTurbo — acceso privado desde tu Samsung

**Esta version evita el inicio de sesión por correo de Cloudflare que daba el error Failed to authenticate.**

Pulsa **▶** en la única celda de código. Reutiliza el servidor ya instalado si sigue vivo. Cuando veas **ENLACE PRIVADO**, abre el enlace completo que aparece debajo. No necesitas ngrok ni correo, ni modificar ninguna contraseña o API Key.

**Seguridad:** El enlace incluye una clave larga de acceso. No lo publiques, no lo compartas ni hagas capturas visibles del enlace. Abre únicamente los enlaces que genere tu propia sesión de Colab. Google Colab puede desconectarse y este acceso es temporal.

Esta versión es experimental. Si aparece un error, comparte solo el mensaje sin contraseñas o enlaces privados.

In [ ]:
# Iniciador privado de MoneyPrinterTurbo. Solo requiere pulsar Ejecutar.
import os
import re
import secrets
import socket
import subprocess
import sys
import time
from pathlib import Path
from urllib.error import HTTPError
from urllib.request import urlopen, urlretrieve

REPO = Path("/content/MoneyPrinterTurbo")
REPO_URL = "https://github.com/Danifloo/MoneyPrinterTurbo.git"
BACKEND_PORT = 8501

def is_server_ready():
    try:
        with urlopen(f"http://127.0.0.1:{BACKEND_PORT}/_stcore/health", timeout=3) as resp:
            return resp.status == 200
    except Exception:
        return False

# Reutiliza MoneyPrinterTurbo si ya fue instalado.
if not is_server_ready():
    if not (REPO / "webui" / "Main.py").is_file():
        if REPO.exists():
            raise RuntimeError("Carpeta MoneyPrinterTurbo incompleta. Revisa la carpeta en Colab.")
        print("Descargando MoneyPrinterTurbo...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)
    if not (REPO / ".venv" / "bin" / "python").exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
        subprocess.run(["uv", "python", "install", "3.11"], check=True, cwd=REPO)
        subprocess.run(["uv", "sync", "--frozen", "--python", "3.11"], check=True, cwd=REPO)
    print("Iniciando servidor de videos...")
    server_log = Path("/content/mpt-servidor.log")
    log_handle = server_log.open("a")
    mpt_server = subprocess.Popen(
        ["uv", "run", "streamlit", "run", "webui/Main.py",
         f"--server.port={BACKEND_PORT}", "--server.address=127.0.0.1",
         "--server.headless=true", "--browser.gatherUsageStats=false"],
        cwd=REPO, stdout=log_handle, stderr=subprocess.STDOUT
    )
    for _ in range(60):
        if is_server_ready():
            break
        if mpt_server.poll() is not None:
            raise RuntimeError("MoneyPrinterTurbo se cerro. Registro:\n" + server_log.read_text(errors="replace")[-2000:])
        time.sleep(2)
    else:
        raise RuntimeError("Servidor sin respuesta. Registro:\n" + server_log.read_text(errors="replace")[-2000:])

print("Servidor de videos: funcionando.")

# Un proxy propio restringe todas las paginas y las conexiones WebSocket
# mediante un enlace de acceso privado y una cookie segura. No usa Cloudflare Access.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "aiohttp"], check=True)
proxy_script = Path("/content/mpt_private_proxy.py")
urlretrieve(
    "https://raw.githubusercontent.com/Danifloo/MoneyPrinterTurbo/main/docs/secure_colab_proxy.py",
    proxy_script
)
with socket.socket() as sock:
    sock.bind(("127.0.0.1", 0))
    proxy_port = sock.getsockname()[1]

session_secret = secrets.token_urlsafe(32)
login_token = secrets.token_urlsafe(32)
proxy_env = os.environ.copy()
proxy_env.update({
    "MPT_SESSION_SECRET": session_secret,
    "MPT_LOGIN_TOKEN": login_token,
    "MPT_PROXY_PORT": str(proxy_port),
    "MPT_BACKEND": f"http://127.0.0.1:{BACKEND_PORT}"
})
del session_secret
proxy_log = Path("/content/mpt-private-proxy.log")
mpt_proxy = subprocess.Popen(
    [sys.executable, str(proxy_script)],
    env=proxy_env, stdout=proxy_log.open("w"), stderr=subprocess.STDOUT
)
for _ in range(30):
    time.sleep(0.5)
    if mpt_proxy.poll() is not None:
        raise RuntimeError("Error en el acceso privado:\n" + proxy_log.read_text(errors="replace")[-2000:])
    try:
        urlopen(f"http://127.0.0.1:{proxy_port}/", timeout=2)
    except HTTPError as exc:
        if exc.code == 401:
            break
    except Exception:
        pass
else:
    raise RuntimeError("No responde el proxy:\n" + proxy_log.read_text(errors="replace")[-2000:])

binary = Path("/content/cloudflared")
if not binary.is_file():
    print("Instalando conexion segura...")
    urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        binary
    )
    binary.chmod(0o755)

# Cloudflare Quick Tunnel solo transporta HTTPS. La autenticacion se hace en nuestro proxy.
# Importante: quitar --allowed-mail evita el fallo del acceso por email de Cloudflare.
tunnel_log = Path("/content/mpt-tunnel.log")
mpt_tunnel = subprocess.Popen(
    [str(binary), "tunnel", "--protocol", "http2",
     "--url", f"http://127.0.0.1:{proxy_port}"],
    stdout=tunnel_log.open("w"), stderr=subprocess.STDOUT
)
pattern = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
for _ in range(75):
    time.sleep(2)
    output = tunnel_log.read_text(errors="replace")
    match = pattern.search(output)
    if match:
        print("\n✅ ENLACE PRIVADO PARA ABRIR MONEYPRINTERTURBO:\n")
        print(match.group(0) + "/__mpt_private_login?token=" + login_token)
        print("\nPulsa el enlace completo. No hace falta introducir correo.")
        print("IMPORTANTE: Quien tenga el enlace completo puede entrar. No lo compartas.")
        print("Guarda tus videos; Colab y su enlace son temporales.")
        break
    if mpt_tunnel.poll() is not None:
        raise RuntimeError("Fallo la conexion remota:\n" + output[-2200:])
else:
    raise RuntimeError("No se creo el enlace privado. Registro:\n" + output[-2200:])